# AnemiaScan v3: Palm photo + Nail photo + gender + age → fused Hb → Anemia

1. Runtime → **Run all**. Allow Google Drive access (use the **capstone** Google account, where `FINAL_models/` is).
2. In the last cell: upload the **nail photo**, then the **palm photo** of the same person, then answer **gender** and **age range**.
3. You get both detected regions, the palm Hb, the nail Hb, the **fused Hb** and the anemia verdict.

**How the two are combined:** a weighted average with the paper's inverse-MSE weights, **palm 0.503 and nail 0.497**. If one region is not found, the result uses the other one alone. If palm and nail differ by more than 1.5 g/dL, you get a warning to retake the photos.

**From v2:** gender and age go into both Hb models, and the WHO cut-off follows the sex: **women < 12 g/dL, men < 13 g/dL**. The other clinical values are still filled with typical dataset values.

Models (searched in this order: `FINAL_models/1_Nail/models`, `FINAL_models/2_Palm/models`, `HbPrediction_Models/`, `MyDrive/`):
- **Nail:** `best_nail_seg_model.pth` (DeepLabV3-ResNet18) + `hb_model.pkl`, `hb_scaler.pkl`, `hb_features.pkl`
- **Palm:** `best_palm_seg_model.pth` + `palm_seg_config.json` (DeepLabV3-ResNet18) + `palm_hb_anemia_models.pkl`

*Research prototype, not a medical diagnosis.*

In [ ]:
# ── 1. Setup and load both models (run once) ──────────────────────────────
from google.colab import drive, files
drive.mount('/content/drive')
!pip install -q segmentation-models-pytorch

import os, json, pickle, warnings
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import segmentation_models_pytorch as smp
warnings.filterwarnings("ignore")

DRIVE = "/content/drive/MyDrive"
SEARCH_DIRS = [f"{DRIVE}/FINAL_models/1_Nail/models", f"{DRIVE}/FINAL_models/2_Palm/models",
               f"{DRIVE}/HbPrediction_Models", DRIVE]

def find_file(name):
    for d in SEARCH_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found in: " + ", ".join(SEARCH_DIRS))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# WHO hemoglobin cut-offs for non-pregnant adults (g/dL)
CUTOFF = {"female": 12.0, "male": 13.0}

def who_status(hb, sex):
    if hb >= CUTOFF[sex]: return False, "Normal"
    if hb >= 11.0:        return True, "Mild anemia"
    if hb >= 8.0:         return True, "Moderate anemia"
    return True, "Severe anemia"

# Age choices → value given to the model (the models were trained on young adults, mostly 18-25 years)
AGE_RANGES = [("Under 18", 17.0), ("18-19", 18.5), ("20-21", 20.5), ("22-23", 22.5),
              ("24-25", 24.5), ("26-30", 28.0), ("Above 30", 30.0)]

def sex_codes(threshold_values, female_code=1):
    """The two Hb models were trained with different sex codes (female = 1, male = 0 or 2).
    Read the split thresholds the forest learned on the sex column to find the male code."""
    t = np.asarray(threshold_values, dtype=float)
    male = 2 if (t.size and t.max() > 1.0) else 0
    return {"female": female_code, "male": male}

# ════════════════════════ NAIL ════════════════════════
nail_seg = smp.DeepLabV3(encoder_name="resnet18", encoder_weights=None, in_channels=3, classes=2)
nail_seg.load_state_dict(torch.load(find_file("best_nail_seg_model.pth"), map_location=device))
nail_seg.to(device).eval()
NAIL_SIZE, NAIL_THRESHOLD = 512, 0.2

nail_hb     = pickle.load(open(find_file("hb_model.pkl"), "rb"))
nail_scaler = pickle.load(open(find_file("hb_scaler.pkl"), "rb"))
NAIL_FEATS  = pickle.load(open(find_file("hb_features.pkl"), "rb"))
NAIL_COLOR_COLS = ["rgb_r_mean", "rgb_g_mean", "rgb_b_mean", "rgb_r_std", "rgb_g_std", "rgb_b_std",
                   "lab_l_mean", "lab_a_mean", "lab_b_mean", "lab_l_std", "lab_a_std", "lab_b_std",
                   "hsv_h_mean", "hsv_s_mean", "hsv_v_mean", "hsv_h_std", "hsv_s_std", "hsv_v_std",
                   "ratio_rg", "ratio_rb", "ratio_gb", "r_p25", "r_p75", "g_p25", "g_p75", "b_p25", "b_p75"]
NAIL_META_COLS = [c for c in NAIL_FEATS if c not in NAIL_COLOR_COLS]
NAIL_SEX_COL = next(c for c in NAIL_META_COLS if c.lower() in ("sexo", "sex"))
NAIL_AGE_COL = next(c for c in NAIL_META_COLS if c.lower() in ("edad", "age"))

# The nail model also takes metadata (sex, age, symptoms ...). No questions are asked in this
# version, so typical (median) values from the dataset are used.
try:
    _df = pd.read_csv(find_file("final_hb_dataset.csv"))
    _df = _df[[c for c in NAIL_META_COLS if c in _df.columns]].replace("?", np.nan).apply(pd.to_numeric, errors="coerce")
    NAIL_DEFAULTS = _df.median().to_dict()
except Exception:
    NAIL_DEFAULTS = {}
    print("⚠️ final_hb_dataset.csv not found: nail metadata set to 0")

def nail_color_features(roi_bgr):
    rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB); lab = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2LAB); hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    m = rgb.astype(np.int32).sum(axis=2) > 10
    if m.sum() < 100:
        return None
    rgb, lab, hsv = rgb[m].astype(np.float32), lab[m].astype(np.float32), hsv[m].astype(np.float32)
    r, g, b = rgb[:, 0], rgb[:, 1], rgb[:, 2]
    vals = [r.mean(), g.mean(), b.mean(), r.std(), g.std(), b.std(),
            lab[:, 0].mean(), lab[:, 1].mean(), lab[:, 2].mean(), lab[:, 0].std(), lab[:, 1].std(), lab[:, 2].std(),
            hsv[:, 0].mean(), hsv[:, 1].mean(), hsv[:, 2].mean(), hsv[:, 0].std(), hsv[:, 1].std(), hsv[:, 2].std(),
            r.mean() / (g.mean() + 1e-6), r.mean() / (b.mean() + 1e-6), g.mean() / (b.mean() + 1e-6),
            np.percentile(r, 25), np.percentile(r, 75), np.percentile(g, 25), np.percentile(g, 75),
            np.percentile(b, 25), np.percentile(b, 75)]
    return dict(zip(NAIL_COLOR_COLS, map(float, vals)))

@torch.no_grad()
def segment_nail(img_bgr):
    H, W = img_bgr.shape[:2]
    x = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), (NAIL_SIZE, NAIL_SIZE)) / 255.0
    x = torch.tensor(x.transpose(2, 0, 1), dtype=torch.float32)[None].to(device)
    prob = torch.softmax(nail_seg(x), dim=1)[0, 1].cpu().numpy()
    mask = cv2.resize((prob > NAIL_THRESHOLD).astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return mask, None, None
    box = cv2.boundingRect(max(cnts, key=cv2.contourArea))          # largest nail is used for Hb
    x0, y0, w, h = box
    return mask, (img_bgr * mask[:, :, None])[y0:y0 + h, x0:x0 + w], box

# sex column of the nail model and its codes (thresholds are stored on the scaled values → unscale them)
_i = NAIL_FEATS.index(NAIL_SEX_COL)
_t = np.concatenate([e.tree_.threshold[e.tree_.feature == _i] for e in nail_hb.estimators_])
NAIL_SEX = sex_codes(_t * nail_scaler.scale_[_i] + nail_scaler.mean_[_i])

def predict_nail(roi_bgr, sex, age):
    feats = nail_color_features(roi_bgr)
    if feats is None:
        return None
    row = {c: NAIL_DEFAULTS.get(c, 0.0) for c in NAIL_META_COLS}
    row[NAIL_SEX_COL], row[NAIL_AGE_COL] = NAIL_SEX[sex], age
    row.update(feats)
    X = nail_scaler.transform(np.array([[row[c] for c in NAIL_FEATS]], dtype=np.float64))
    return float(nail_hb.predict(X)[0])

# ════════════════════════ PALM ════════════════════════
palm_cfg = json.load(open(find_file("palm_seg_config.json")))
palm_seg = getattr(smp, palm_cfg["arch"])(encoder_name=palm_cfg["encoder"], encoder_weights=None, in_channels=3, classes=2)
palm_seg.load_state_dict(torch.load(find_file("best_palm_seg_model.pth"), map_location=device))
palm_seg.to(device).eval()
PALM_SIZE = palm_cfg["img_size"]

PALM = pickle.load(open(find_file("palm_hb_anemia_models.pkl"), "rb"))
PALM_COLOR_FEATS = PALM["color_features"]

# Compatibility: a model pickled with an older scikit-learn can miss an attribute that newer
# SimpleImputer versions expect. Add it back so the saved pipeline still runs.
from sklearn.impute import SimpleImputer
for _step in getattr(PALM["hb_model"], "steps", []):
    if isinstance(_step[1], SimpleImputer) and not hasattr(_step[1], "_fill_dtype"):
        _step[1]._fill_dtype = _step[1].statistics_.dtype

def palm_color_features(roi_bgr, max_side=512):
    h, w = roi_bgr.shape[:2]; k = max_side / max(h, w)
    if k < 1:
        roi_bgr = cv2.resize(roi_bgr, (int(w * k), int(h * k)), interpolation=cv2.INTER_AREA)
    rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB); lab = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2LAB); hsv = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2HSV)
    m = (rgb.astype(np.int32).sum(axis=2) > 30).astype(np.uint8)
    m = cv2.erode(m, np.ones((5, 5), np.uint8)).astype(bool)
    if m.sum() < 500:
        return None
    p_rgb, p_lab, p_hsv = rgb[m].astype(np.float32), lab[m].astype(np.float32), hsv[m].astype(np.float32)
    r, g, b = p_rgb.T; eps = 1e-6; f = {}
    for name, arr in [("rgb_r", r), ("rgb_g", g), ("rgb_b", b), ("lab_l", p_lab[:, 0]), ("lab_a", p_lab[:, 1]),
                      ("lab_b", p_lab[:, 2]), ("hsv_h", p_hsv[:, 0]), ("hsv_s", p_hsv[:, 1]), ("hsv_v", p_hsv[:, 2])]:
        f[f"{name}_mean"], f[f"{name}_std"] = arr.mean(), arr.std()
    f.update(ratio_rg=r.mean() / (g.mean() + eps), ratio_rb=r.mean() / (b.mean() + eps), ratio_gb=g.mean() / (b.mean() + eps))
    for name, arr in [("rgb_r", r), ("rgb_g", g), ("rgb_b", b), ("lab_a", p_lab[:, 1]), ("hsv_s", p_hsv[:, 1])]:
        f[f"{name}_p25"], f[f"{name}_p75"] = np.percentile(arr, 25), np.percentile(arr, 75)
    return {k: float(f[k]) for k in PALM_COLOR_FEATS}

@torch.no_grad()
def segment_palm(img_bgr, threshold=0.5):
    H, W = img_bgr.shape[:2]; k = PALM_SIZE / max(H, W)
    nh, nw = int(round(H * k)), int(round(W * k))
    canvas = np.zeros((PALM_SIZE, PALM_SIZE, 3), np.uint8)
    canvas[:nh, :nw] = cv2.resize(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), (nw, nh), interpolation=cv2.INTER_AREA)
    x = torch.from_numpy(canvas.transpose(2, 0, 1)).float()[None].to(device) / 255.0
    prob = torch.softmax(palm_seg(x), 1)[0, 1, :nh, :nw].cpu().numpy()
    mask = cv2.resize((prob > threshold).astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return mask, None, None
    big = max(cnts, key=cv2.contourArea)
    mask = np.zeros_like(mask); cv2.drawContours(mask, [big], -1, 1, cv2.FILLED)
    box = cv2.boundingRect(big); x0, y0, w, h = box
    return mask, (img_bgr * mask[:, :, None])[y0:y0 + h, x0:x0 + w], box

_rf = PALM["hb_model"].steps[-1][1] if hasattr(PALM["hb_model"], "steps") else PALM["hb_model"]
_i = PALM["hb_features"].index("sex")
PALM_SEX = sex_codes(np.concatenate([e.tree_.threshold[e.tree_.feature == _i] for e in _rf.estimators_]),
                     PALM.get("female_code", 1))

def predict_palm(roi_bgr, sex, age):
    feats = palm_color_features(roi_bgr)
    if feats is None:
        return None
    feats = dict(feats, sex=PALM_SEX[sex], age=age)
    # the other clinical columns are left empty; the saved pipeline fills them with the training medians
    X = np.array([[feats.get(c, np.nan) for c in PALM["hb_features"]]], dtype=float)
    return float(PALM["hb_model"].predict(X)[0])

# ════════════════════════ FUSION ════════════════════════
# Inverse-MSE weights from the paper (palm MSE 0.872, nail MSE 0.881 on the development split)
FUSION_WEIGHTS = {"palm": 0.503, "nail": 0.497}
DISAGREE_WARN = 1.5   # g/dL gap between the palm and nail estimates that triggers a warning

def fuse(hb_palm, hb_nail):
    preds = {k: v for k, v in (("palm", hb_palm), ("nail", hb_nail)) if v is not None}
    if not preds:
        return None
    return sum(FUSION_WEIGHTS[k] * v for k, v in preds.items()) / sum(FUSION_WEIGHTS[k] for k in preds)

# ════════════════════════ helpers ════════════════════════
def upload_image(prompt):
    print(prompt)
    up = files.upload()
    if not up:
        raise SystemExit("No file uploaded.")
    name = list(up.keys())[0]
    img = cv2.imdecode(np.frombuffer(up[name], np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise SystemExit("Could not read that image. Use a JPG or PNG.")
    return name, img

def ask_sex():
    while True:
        a = input("Gender?  1 = Female   2 = Male  : ").strip().lower()
        if a in ("1", "f", "female"): return "female"
        if a in ("2", "m", "male"):   return "male"
        print("Please type 1 or 2.")

def ask_age():
    print("Age range:")
    for i, (label, _) in enumerate(AGE_RANGES, 1):
        print(f"   {i} = {label}")
    while True:
        a = input("Type the number of the age range : ").strip()
        if a.isdigit() and 1 <= int(a) <= len(AGE_RANGES):
            return AGE_RANGES[int(a) - 1]
        print(f"Please type a number from 1 to {len(AGE_RANGES)}.")

def roi_overlay(img_bgr, mask, box, color=(46, 204, 113)):
    """Input photo with the detected region tinted, outlined and boxed."""
    out = img_bgr.copy()
    tint = np.zeros_like(out); tint[:] = color
    m = mask.astype(bool)
    out[m] = cv2.addWeighted(out, 0.55, tint, 0.45, 0)[m]
    t = max(2, int(round(max(img_bgr.shape[:2]) / 300)))
    cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, cnts, -1, color, t)
    if box is not None:
        x0, y0, w, h = box
        cv2.rectangle(out, (x0, y0), (x0 + w, y0 + h), (0, 0, 255), t)
    return out

def show_combined(rows, hb, anemic, severity, sex, age_label, hb_palm, hb_nail):
    """rows = [(region, img, mask, roi, box), …] for palm and nail."""
    fig = plt.figure(figsize=(19, 9.5))
    gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 0.8, 1.1])
    for r, (region, img, mask, roi, box) in enumerate(rows):
        a = fig.add_subplot(gs[r, 0]); a.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a.set_title(f"{region.capitalize()} photo"); a.axis("off")
        a = fig.add_subplot(gs[r, 1]); a.axis("off")
        if roi is not None:
            a.imshow(cv2.cvtColor(roi_overlay(img, mask, box), cv2.COLOR_BGR2RGB)); a.set_title(f"Detected {region} (region of interest)")
        else:
            a.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a.set_title(f"No {region} detected", color="crimson")
        a = fig.add_subplot(gs[r, 2]); a.axis("off")
        if roi is not None:
            a.imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)); a.set_title(f"{region.capitalize()} region used for Hb")
    ax = fig.add_subplot(gs[:, 3]); ax.axis("off")
    c = "crimson" if anemic else "green"
    fmt = lambda v: "not detected" if v is None else f"{v:.2f} g/dL"
    ax.text(0.5, 0.86, f"{hb:.1f} g/dL", ha="center", fontsize=34, fontweight="bold", color=c)
    ax.text(0.5, 0.76, "ANEMIC" if anemic else "NOT ANEMIC", ha="center", fontsize=23, fontweight="bold", color=c)
    ax.text(0.5, 0.70, severity, ha="center", fontsize=15, color=c)
    ax.text(0.5, 0.58, f"Palm: {fmt(hb_palm)}   (weight {FUSION_WEIGHTS['palm']})", ha="center", fontsize=12, color="dimgray")
    ax.text(0.5, 0.53, f"Nail: {fmt(hb_nail)}   (weight {FUSION_WEIGHTS['nail']})", ha="center", fontsize=12, color="dimgray")
    ax.text(0.5, 0.42, f"{sex.capitalize()}, age {age_label}", ha="center", fontsize=12, color="dimgray")
    ax.text(0.5, 0.37, f"WHO: anemic if Hb < {CUTOFF[sex]:g} g/dL ({'women' if sex == 'female' else 'men'})", ha="center", fontsize=11, color="gray")
    ax.text(0.5, 0.29, "Research prototype, not a medical diagnosis", ha="center", fontsize=10, color="gray")
    plt.suptitle("AnemiaScan v3: palm + nail + gender + age → fused Hb → Anemia", fontsize=15, fontweight="bold")
    plt.tight_layout(); plt.show()

print(f"✅ Models loaded (v3)  |  sex codes nail {NAIL_SEX}, palm {PALM_SEX}  |  nail Hb: {type(nail_hb).__name__}  |  palm Hb: {PALM['hb_model_name']}  |  palm seg IoU {palm_cfg.get('best_val_iou')}")

## 2. Screen a patient
Run this cell → upload the **nail** photo → upload the **palm** photo → choose **gender** → choose **age range**. **Re-run it for another person.**

In [ ]:
# ── 2. Upload nail + palm → gender + age → fused result ───────────────────
nail_name, nail_img = upload_image("💅 Upload the NAIL photo:")
palm_name, palm_img = upload_image("✋ Upload the PALM photo:")
sex = ask_sex()
age_label, age = ask_age()

nail_mask, nail_roi, nail_box = segment_nail(nail_img)
palm_mask, palm_roi, palm_box = segment_palm(palm_img)
hb_nail = predict_nail(nail_roi, sex, age) if nail_roi is not None else None
hb_palm = predict_palm(palm_roi, sex, age) if palm_roi is not None else None

hb = fuse(hb_palm, hb_nail)
if hb is None:
    raise SystemExit("❌ Neither a nail nor a palm was detected. Retake both photos closer, in good light.")
anemic, severity = who_status(hb, sex)

print("\n" + "=" * 56)
print(f"  Photos               : {nail_name} (nail), {palm_name} (palm)")
print(f"  Gender / age         : {sex.capitalize()}, {age_label}")
print(f"  Palm Hb              : {'not detected' if hb_palm is None else f'{hb_palm:.2f} g/dL'}")
print(f"  Nail Hb              : {'not detected' if hb_nail is None else f'{hb_nail:.2f} g/dL'}")
print(f"  Fused hemoglobin     : {hb:.2f} g/dL")
print(f"  Verdict              : {'ANEMIC' if anemic else 'NOT ANEMIC'} ({severity}; WHO cut-off {CUTOFF[sex]:g} g/dL)")
print("=" * 56)
if hb_palm is None: print("  ⚠️ No palm detected, so the result uses the nail only.")
if hb_nail is None: print("  ⚠️ No nail detected, so the result uses the palm only.")
if hb_palm is not None and hb_nail is not None and abs(hb_palm - hb_nail) > DISAGREE_WARN:
    print(f"  ⚠️ Palm and nail differ by {abs(hb_palm - hb_nail):.1f} g/dL. Retake the photos for a reliable result.")
if age_label in ("Under 18", "Above 30"):
    print("  ⚠️ The models were trained on young adults (mostly 18-25), so this result is less reliable.")
show_combined([("palm", palm_img, palm_mask, palm_roi, palm_box), ("nail", nail_img, nail_mask, nail_roi, nail_box)],
              hb, anemic, severity, sex, age_label, hb_palm, hb_nail)